# Advanced document indexing

# Splitting and ingesting HTML content

## Splitting and ingesting the content of a single URL (on Cornwall)

### Preparing the Chroma DB collections

In [8]:
# header_template полностью заменяет дефолт — убираем подозрительный Referer
WIKIMEDIA_HEADERS = {
    "User-Agent": "MyRAGStudy/1.0 (https://github.com/yourname; your@email.com) python-requests/2.31.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip",
}

In [9]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import getpass

OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

In [10]:
corwnall_granular_collection = Chroma( #A
    collection_name="cornwall_granular",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

corwnall_granular_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

In [11]:
corwnall_coarse_collection = Chroma( #A 
    collection_name="cornwall_coarse",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

corwnall_coarse_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

### Loading the HTML content with the AsyncHtmlLoader 

In [12]:
from langchain_community.document_loaders import AsyncHtmlLoader

In [13]:
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url, header_template=WIKIMEDIA_HEADERS)
docs = html_loader.load()

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.82it/s]


In [14]:
len(docs)

1

In [15]:
docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content='<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">\n<head>\n<meta charset="UTF-8">\n<title>Cornwall – Travel guide at Wikivoyage</title>\n<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-f

### Splitting into granular chunks with the HTMLSectionSplitter 

In [16]:
from langchain_text_splitters import HTMLSectionSplitter

In [17]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)

In [18]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(html_string) #B
        all_chunks.extend(temp_chunks) 

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

In [19]:
granular_chunks = split_docs_into_granular_chunks(docs)

#### Ingesting granular chunks

In [20]:
corwnall_granular_collection.add_documents(documents=granular_chunks)

['9aae1fbb-6cde-4caf-82af-eed098e65632',
 'd5e8b8a7-da70-43c4-aeb6-92048dc13cb4',
 '867249d5-f46b-4652-a9d0-632fd65306e5',
 '72103d2a-ff47-4e1a-ad54-67dc7dd97189',
 'b1155913-869e-469c-b1e5-40b0f23008cd',
 '5845730f-e572-402c-84d5-2bb511fa0cf1',
 '6cebaf66-2a7e-48f0-a08a-2318e5034e83',
 '38f8a2ef-e42c-4de2-9039-ac4711b4efd8',
 'e6b1bcb6-a772-4de8-ad1a-4014566f2c43',
 'd83b9f6d-d572-4c8f-8d5c-4dee95964fe5',
 '42451d07-e2ef-4407-8291-cf653da9d0fe',
 '9e0d4d2a-0457-49e7-a5db-bec3666c67e9',
 '841c455f-ac4f-4c28-b0c3-6ae859462899',
 '00de31ff-5699-45d0-97a3-2ad52696f3c1',
 '79f1919a-0ed6-4cc0-890e-d5a57dad85b9',
 '609b773d-244b-4627-ac45-79621ba7acaf',
 '9ab5e8c9-e236-47e0-acb7-a6a52ec17762',
 'b92a3b89-5340-461a-8a35-fcaa609f5234',
 'e756379c-6b66-4f8c-a1e3-f60c57a8f0ff']

#### Searching granular chunks

In [21]:
results = corwnall_granular_collection.similarity_search(query='Events or festivals in Cornwall',k=3)
for doc in results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance 

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter 

In [22]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [23]:
html2text_transformer = Html2TextTransformer()

In [24]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=300)

In [25]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs) #A 
    coarse_chunks = text_splitter.split_documents(text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

In [26]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

#### Ingesting coarse chunks

In [27]:
corwnall_coarse_collection.add_documents(documents=coarse_chunks)

['f6a19280-26c6-4d34-9855-4e1bc3cc8d8c',
 'b0c159ac-661d-4816-91cc-b9eb75141da0',
 'b2d0f5be-1b67-4bde-b52f-37181f488c2a',
 '81ff3764-de6e-4723-a83f-5aa87a0e8256',
 '9e65b6d5-f610-40df-b560-d9b8ee20c6be',
 '5c9587c5-09ed-4edd-914b-86613fed191f',
 '30c46cba-8b2f-4f81-b1d1-e2670211309b',
 'dbc00dbe-89d8-4932-acb8-9439b9bad8a1',
 'c4576bcf-2431-4645-b393-6192340faa8f',
 'bd5c1958-3262-4359-bb75-5cfca1236b6a',
 '23625e46-64e3-4e5a-9b61-3ab09518663a',
 '62572db6-b0fb-4f09-a6dd-f990d5e48a55',
 '8e363752-8c82-4c88-a264-03a8ae4f5481',
 '20cdc9c3-8f31-4fad-973d-c0849179aacd',
 '62edfe65-0a4d-4a23-b9f6-45d85f6d534e']

#### Searching coarse chunks

In [28]:
results = corwnall_coarse_collection.similarity_search(query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corni

## Splitting and ingesting the content of various URLs (across UK destinations)

### Preparing the Chroma DB collections

In [29]:
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

uk_granular_collection.reset_collection() #B

In [30]:
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter 

In [31]:
# Reduce this list if you want to save on processing fees
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [32]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

In [33]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url, header_template=WIKIMEDIA_HEADERS) #C
    docs =  html_loader.load() #D
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists 
#C Loader for one destination
#D Documents of one destination 

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  7.20it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.38it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.25it/s]


...

{'source': 'https://en.wikivoyage.org/wiki/Rye_(England)', 'title': 'Rye (England) – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.79it/s]


{'source': 'https://en.wikivoyage.org/wiki/Seaford', 'title': 'Seaford – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.25it/s]


{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest', 'title': 'Ashdown Forest – Travel guide at Wikivoyage', 'language': 'en'}


#### Searching 

In [34]:
granular_results = uk_granular_collection.similarity_search(query="Events or festivals in East Sussex",k=4)
for doc in granular_results:
    print(doc)

page_content='Brighton' metadata={'Header 1': 'Brighton'}
page_content='South Cornwall' metadata={'Header 1': 'South Cornwall'}
page_content='Seaford' metadata={'Header 1': 'Seaford'}
page_content='Penzance' metadata={'Header 1': 'Penzance'}


In [35]:
coarse_results = uk_coarse_collection.similarity_search(query="Events or festivals in East Sussex",k=4)
for doc in coarse_results:
    print(doc)

page_content='### Events

[edit]

A market during the Brighton Festival. The Theatre Royal is the red building.
A colourful parade down Queens Road during Pride in 2016.

  * **Brighton Racecourse** has flat-racing April-Oct. It's on Freshfield Rd a mile east of town centre.
  * **Plumpton Racecourse** is National Hunt (jumps races) Nov-March, but it's 10 mi (16 km) north in Lewes.
  * Brighton Festival Fringe: early May – early June, ☏ +44 1273 764900, info@brightonfringe.org. The Fringe runs at the same time as the main festival, and features over 600 events, including comedy, theatre, music, and "open houses" (local artists exhibiting in their own homes) and tours (haunted pubs, Regency Brighton, churches, cemeteries, sewers, etc.)_(date needs fixing)_
  * Brighton Festival: May, ☏ +44 1273 709709 (tickets), tickets@brightonfestival.org. The Brighton Festival, in May each year, is the second biggest arts festival in Great Britain (coming closely behind Edinburgh). Music of all sorts

In [36]:
granular_results = uk_granular_collection.similarity_search(
    query="Beaches in Conrwall",k=4)
for doc in granular_results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='North Cornwall' metadata={'Header 1': 'North Cornwall'}
page_content='West Cornwall' metadata={'Header 1': 'West Cornwall'}
page_content='South Cornwall' metadata={'Header 1': 'South Cornwall'}


In [37]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Beaches in Cornwall",k=4)
for doc in coarse_results:
    print(doc)

page_content='**North Cornwall** is in Cornwall. It includes much of the Cornish coast along
the Celtic Sea and some top surfing areas.

## Towns and villages

[edit]

Map of North Cornwall

  * 50.466-4.7181 Bodmin \- Former administrative centre of Cornwall and home of Bodmin Jail
  * 50.422-4.6712 Lostwithiel — home to Restormel Castle, notable for being almost completely cylindrical
  * 50.824-4.5423 Bude — fishing village and holiday resort on the north coast of Cornwall
  * 50.635-4.3544 Launceston — home of the 11th-century Launceston Castle, and the Launceston narrow-gauge heritage steam railway
  * 50.412-5.07575 Newquay — former fishing village, now surf capital of the UK and home to Cornwall's principal airport
  * 50.538-4.9386 Padstow — fishing village and holiday resort on Cornwall's north coast
  * 50.3437-5.15467 Perranporth — a seaside resort town backed by extensive sand dunes which reach nearly a mile inland
  * 50.432-4.948 St Columb Major — one of only two places w

# Embedding strategy

## Embedding child chunks with ParentDocumentRetriever

In [38]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Setting up the Parent Document retriever

In [39]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [40]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination 
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  9.88it/s]
c:\Users\user\Desktop\building-llm-applications\.full-env\Lib\site-packages\langchain_community\document_loaders\async_html.py:215: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, parser)


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.63it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.65it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)



Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.32it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.86it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [41]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['47677186-6fc5-4ea1-83ea-aebb76180c7f',
 'd45f4294-f8f7-426d-8941-66d658560fa1',
 '73d46982-11e1-42de-ace9-a20ef85b75de',
 'b988e87d-54e4-4fc0-aaa9-e0c2c3211fd2',
 '7c548414-95d4-48fd-8b7b-470981b3f8eb',
 'dece5059-f8a2-441f-a408-3fbad618560b',
 '7974cbc1-95c7-4052-9c58-4e20329ae47d',
 '04bed1ae-6e8e-4a87-a0c4-b8af2e742ffe',
 'bf67fd2a-5641-457f-aac5-47790f4038e3',
 '7b4f399e-fad2-4dc8-9dec-c2720e7550b1',
 '9fe6be10-5baf-4136-a8ba-eaddd4c5d6ce',
 'ce4eb848-3328-4633-bf6e-3a0f0a03a895',
 'eef9e722-4244-4f03-9e31-338ea9d12c38',
 '38d13d6e-4483-4429-8587-143868cbe83d',
 'e350c980-fd3d-4b6f-a86f-5a81625b8424',
 '3753f27a-d512-4dad-a1c6-53dca52482cd',
 'd06cb81e-c15e-4c9a-98a2-9fd4b6b6afc5',
 '596a808e-38c6-40a0-80ba-9c24cf5f8163',
 'ee5af3f5-86f2-4b62-b6a5-ea16cdb4d710',
 '007975c3-3e21-40fd-91df-77d805e47ede']

### Performing a search on granular information 

In [42]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [43]:
len(retrieved_docs)

4

In [44]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)')

### Comparing with direct semantic search on child chunks

In [45]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [46]:
len(child_docs_only)

4

In [47]:
child_docs_only[0]

Document(id='75118978-f685-455e-af0e-a5fa80a13455', metadata={'doc_id': '47677186-6fc5-4ea1-83ea-aebb76180c7f', 'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)')

In [41]:
# IMPORTANT: as you can see a granular search would have identified the chunk, but it would have lost the usefulcontext about travelling in Cornwall

## Embedding child chunks with MultiVectorRetriever

In [48]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [49]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(
        openai_api_key=OPENAI_API_KEY),
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [50]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]) #F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id #G

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_granular_chunks) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into parent coarse chunks
#E Iterate over the parent coarse chunks
#F Create child granular chunks form each parent coarse chunk
#G Link each child granular chunk to its parent coarse chunk
#H Ingest the child granular chunks into the vector store
#I Ingest the parent coarse chunks into the document store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  8.71it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.63it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.64it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.25it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 10.82it/s]

Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [51]:
retrieved_docs = multi_vector_retriever.invoke(
    "Cornwall Ranger")

In [52]:
len(retrieved_docs)

4

In [53]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)')

In [61]:
##IMPORTANT: same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks

### Comparing with direct semantic search on child chunks

In [54]:
child_docs_only =  child_chunks_collection.similarity_search(
    "Cornwall Ranger")

In [55]:
len(child_docs_only)

4

In [56]:
child_docs_only[0]

Document(id='600e3efe-5ff1-4868-962a-5371a03bf000', metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'doc_id': '59629f62-81e1-4eb5-894b-7611630f10dc'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)')

In [65]:
## IMPORTANT: Same as before

## Embedding summaries with MultiVectorRetriever

In [57]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

### Setting up the Multi vector retriever (similar to when embedding child chunks)

In [58]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A

summaries_collection = Chroma( #B
    collection_name="uk_summaries",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

summaries_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the summarization chain

In [59]:
llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)

In [60]:
summarization_chain = (
    {"document": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}") #B
    | llm
    | StrOutputParser())

#A Grab the text content from the document
#B Instantiate a prompt asking to generate summary of the provided text
#C Send the LLM the instantiated prompt 
#D Extract the summary text from the response

### Ingesting the coarse chunks and related summaries into doc and vector store

In [61]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        summary_text =  summarization_chain.invoke(
            coarse_chunk) #F
        summary_doc = Document(page_content=summary_text, 
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc) #G

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a summary for the coarse chunk thorugh the summarization chain
#G Link each summary to its related coarse chunk
#H Ingest the summaries into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  9.97it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.70it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.07it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.56it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.93it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [71]:
# COMMENT: the code above is similar to when ingesting child chunks, but it is slower because of the summarization step
# which invokes the LLM.
# The processing can be speeded up by parallelizing the outer for loop on the destination urls.

### Performing a search on granular information

In [62]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

In [63]:
len(retrieved_docs)

4

In [64]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/West_Cornwall'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Looe'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Brighton'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)')]

### Comparing with direct semantic search on summaries

In [65]:
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

In [66]:
len(summary_docs_only)

4

In [77]:
summary_docs_only

[Document(id='9553aa2d-456e-4a38-8f95-28f072f13992', metadata={'doc_id': '14554480-abf6-4b35-ae7d-8580a27ecc9a'}, page_content="Cornwall offers a diverse array of attractions spanning natural beauty, legends, gardens, historic sites, arts, and heritage, including both independent sites and National Trust properties.\n\n- Natural and legendary sights: King Arthur's Hall and Brown Willy on Bodmin Moor; Dozmary Pool and tales of the Beast of the Moor.\n- Gardens and nature: The Eden Project’s two glass domes; the Lost Gardens of Heligan near Mevagissey.\n- Castles, archaeology, and coastal culture: Tintagel Castle (Arthurian legends and early medieval finds); Minack Theatre (clifftop outdoor theatre and museum); St Michael's Mount.\n- Arts and museums: Tate St Ives (modern art); National Maritime Museum, Falmouth (small-boat collection and other exhibits).\n- Mining and industrial heritage: Historic tin/copper mine sites such as Geevor Tin Mine, Poldark Mine, King Edward Mine, Crown Hill 

In [78]:
# COMMENT: a direct search on summaries retrieves denser information, but it is missing out on useful details. 
# However, you might consider using the summaries directly if after testing they prove adequate.

## Embedding hypothetical questions with MultiVectorRetriever

In [67]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

### Setting up the Multi vector retriever (same as when embedding summaries)

In [68]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

hypothetical_questions_collection = Chroma( #B
    collection_name="uk_hypothetical_questions",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

hypothetical_questions_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the chain to generate hypothetical questions

In [69]:
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""

    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

In [70]:
llm_with_structured_output = ChatOpenAI(
    model="gpt-5-nano", 
    openai_api_key=OPENAI_API_KEY).with_structured_output(
        HypotheticalQuestions
)

In [71]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template( #B
        "Generate a list of exactly 4 hypothetical questions that the below text could be used to answer:\n\n{document_text}"
    )
    | llm_with_structured_output #C
    | (lambda x: x.questions) #D
)

#A Grab the text content from the document
#B Instantiate a prompt asking to generate 4 hypothetical questions on the provided text
#C Invoke the LLM configured to return an object containing the questions as a typed list of strings
#D Grab the list of questions from the response

### Ingesting the coarse chunks and related hypothetical questions into doc and vector store

In [72]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_hypothetical_questions = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        hypothetical_questions = hypothetical_questions_chain.invoke(
            coarse_chunk) #F
        hypothetical_questions_docs = [Document(
            page_content=question, metadata={doc_key: coarse_chunk_id})
                    for question 
                    in hypothetical_questions] #G

        all_hypothetical_questions.extend(hypothetical_questions_docs)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_hypothetical_questions) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a list of hypothetical questions for the coarse chunk thorugh the question generation chain
#G Link each hypothetical question to its related coarse chunk
#H Ingest the hypothetical questions into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  9.27it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.48it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.91it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  9.53it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 10.72it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [85]:
retrieved_docs = multi_vector_retriever.invoke(
    "How can you go to Brighton from London?")

In [86]:
len(retrieved_docs)

4

In [87]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Brighton', 'title': 'Brighton – Travel guide at Wikivoyage', 'language': 'en'}, page_content='Brighton  \n---  \nClimate chart (explanation)  \n| J| F| M| A| M| J| J| A| S| O| N| D  \n---|---|---|---|---|---|---|---|---|---|---|---  \n88 8 3 |  60 8 3 |  51 9 4 |  58 12 6 |  56 16 9 |  50 18 12 |  54 20 14 |  62 21 14 |  67 18 12 |  105 15 9 |  103 11 6 |  97 9 4  \nAverage max. and min. temperatures in °C  \nPrecipitation+Snow totals in mm  \nSource: Wikipedia. Visit the Met Office for a five day forecast.  \n| Imperial conversion  \n---  \nJ| F| M| A| M| J| J| A| S| O| N| D  \n3.5 46 37 |  2.4 46 37 |  2 48 39 |  2.3 54 43 |  2.2 61 48 |  2 64 54 |  2.1 68 57 |  2.4 70 57 |  2.6 64 54 |  4.1 59 48 |  4.1 52 43 |  3.8 48 39  \nAverage max. and min. temperatures in °F  \nPrecipitation+Snow totals in inches  \n  \nThe city is close to London, and is increasingly popular with media and music\ntypes who don\'t want to live in t

### Inspecting possible questions matching our question through semantic search

In [88]:
hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search(
    "How can you go to Brighton from London?")

In [89]:
len(hypothetical_question_docs_only)

4

In [90]:
hypothetical_question_docs_only

[Document(id='399bb54e-88e8-4eb1-b572-d363e8f23006', metadata={'doc_id': 'a1f51d6c-2d27-4a6c-b1a0-4f9918dc1545'}, page_content='How can you travel to Brighton by train from London, and what are the two main railway stations in the city?'),
 Document(id='65d2f229-e7e5-4516-ac09-ea93fd3feb1d', metadata={'doc_id': '53e1031c-3d92-4e7e-9398-1770b215dad1'}, page_content='What transportation options are described for getting to Brighton and for getting around the city?'),
 Document(id='fb27db61-b09a-4280-9473-cfe82759e708', metadata={'doc_id': 'f4b555cf-afe0-43e2-b10a-f59fdab3e2e2'}, page_content='What is the fastest way to travel from Gatwick to Brighton, and how long does it take by train according to the text?'),
 Document(id='320ca9aa-f328-4c06-9d18-00ef76577884', metadata={'doc_id': '79b9f77f-21ab-4ab4-bbf4-4a04d5576cbc'}, page_content='If I want to travel around Brighton all day on buses with one fare, what ticket would I buy and how much would it cost?')]

# Granular chunk expansion with MultiVectorRetriever

In [91]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [92]:
granular_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #A

granular_chunks_collection = Chroma( #B
    collection_name="uk_granular_chunks",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

granular_chunks_collection.reset_collection() #C

expanded_chunk_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)
#A Splitter to generate granular chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host expanded chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Ingesting granular and expanded chunks into doc and vector store

In [93]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs) #D

    expanded_chunk_store_items = []
    for i, granular_chunk in enumerate(
        granular_chunks): #E

        this_chunk_num = i #F
        previous_chunk_num = i-1 #F
        next_chunk_num = i+1 #F
        
        if i==0: #F
            previous_chunk_num = None
        elif i==(len(granular_chunks)-1): #F
            next_chunk_num = None

        expanded_chunk_text = "" #G
        if previous_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                previous_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_text += granular_chunks[
            this_chunk_num].page_content #G
        expanded_chunk_text += "\n"

        if next_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                next_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_id = str(uuid.uuid4()) #H
        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text) #I

        expanded_chunk_store_item = (expanded_chunk_id, 
                                     expanded_chunk_doc)
        expanded_chunk_store_items.append(
            expanded_chunk_store_item)

        granular_chunk.metadata[
            doc_key] = expanded_chunk_id #J
            
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        granular_chunks) #K
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items) #L

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into granular chunks
#E Iterate over the granular chunks
#F determine the index of the current chunk and its previous and next chunks
#G Assemble the text of the expanded chunk by including the previous and next chunk
#H Generate the ID of the expanded chunk
#I Create the expanded chunk document
#J Link each granular chunk to its related expanded chunk
#K Ingest the granular chunks into the vector store
#L Ingest the expanded chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.39it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 12.60it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.08it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 13.77it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 13.99it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [94]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")

In [95]:
len(retrieved_docs)

4

In [96]:
retrieved_docs[0]

Document(metadata={}, page_content="Buses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\nserving a number of other towns on branch lines. For train times and fares\nvisit National Rail Enquiries.\nThe **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.\n\nThe **Lo

### Comparing with direct semantic search on granular chunks

In [97]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [98]:
len(child_docs_only)

4

In [99]:
child_docs_only[0]

Document(id='a18b9a25-f88f-433b-8819-2d80d8a39fcd', metadata={'language': 'en', 'doc_id': 'f69a9d0a-6153-4c74-9f19-83942aeb3876', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.')

In [100]:
# COMMENT: the expanded chunk has more useful context